# Module 12: Automated Data Collection (Web Scraping)
## Objective: Building autonomous bots to acquire live, real-world data.

We move beyond static files. In this module, we will learn to automate browser 
actions with **Selenium**, connect to professional **APIs**, and implement 
**Anti-Blocking** strategies to ensure our data pipelines never break.

### 1. Handling Interactive Websites
Selenium allows us to control a web browser programmatically. This is essential 
for sites where data only appears after a **Click**, a **Scroll**, or a **Login**.

**Note**: You must have a WebDriver (like ChromeDriver) installed to run this locally.

In [2]:
!pip install selenium -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 2.1.1 requires sentencepiece, which is not installed.
botocore 1.27.59 requires urllib3<1.27,>=1.25.4, but you have urllib3 2.6.3 which is incompatible.
requests 2.29.0 requires urllib3<1.27,>=1.21.1, but you have urllib3 2.6.3 which is incompatible.
sphinx 5.0.2 requires docutils<0.19,>=0.14, but you have docutils 0.22.4 which is incompatible.


In [7]:
# !pip install selenium
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
import time
from selenium.webdriver.chrome.options import Options

# 1. Setup Options (Optional: e.g., keeping the browser open)
chrome_options = Options()
chrome_options.add_experimental_option("detach", True) 

# 2. Initialize the Driver 
# Selenium 4 will automatically find/download the driver for you
driver = webdriver.Chrome(options=chrome_options)

# 3. Use the driver
driver.get("https://www.google.com")
print(f"Page Title: {driver.title}")

# driver.quit() # Uncomment to close the browser automatically

Page Title: Google


### 2. Building an Automated Data Pipeline
Instead of scraping HTML, we use `requests` to pull structured JSON data. 
This is the professional standard for "Fresh" data updates.

In [10]:
import requests
import json

def fetch_tavily_search(api_key, query):
    url = "https://api.tavily.com/search"
    
    payload = {
        "api_key": api_key,
        "query": query,
        "search_depth": "basic",  # Changed from "smart" to "basic"
        "include_answer": True
    }
    
    headers = {"Content-Type": "application/json"}
    
    response = requests.post(url, json=payload, headers=headers)
    
    if response.status_code == 200:
        data = response.json()
        return {
            "answer": data.get("answer", "No direct answer found."),
            "results_count": len(data.get("results", []))
        }
    else:
        # This will help you debug if another error occurs
        return f"Error: {response.status_code} - {response.text}"

# Use your key here
API_KEY = "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"
result = fetch_tavily_search(API_KEY, "Agriculture trends in Peshawar 2026")
print(result)

{'answer': 'Wheat production in Pakistan for 2026 is expected to be above average, with a focus on climate-smart innovations to boost agricultural resilience. The Bridging Pakistan Agriculture Investment Conference in January 2026 aims to attract investments for sustainable growth. Current trends show a decline in overall crop production compared to previous years.', 'results_count': 5}


### 3. Staying Under the Radar
If your script sends 100 requests in 1 second, the website will block you. 
We use `time.sleep()` with a "Random Jitter" to look human.

In [5]:
import time
import random

def stealth_scraper(urls):
    for url in urls:
        print(f"Scraping: {url}")
        
        # logic to scrape...
        
        # --- Rate Limiting ---
        # Sleep for a random time between 2 and 5 seconds
        wait_time = random.uniform(2, 5)
        print(f"Waiting for {wait_time:.2f} seconds to avoid detection...")
        time.sleep(wait_time)

stealth_scraper(["site.com/page1", "site.com/page2"])

Scraping: site.com/page1
Waiting for 3.40 seconds to avoid detection...
Scraping: site.com/page2
Waiting for 2.91 seconds to avoid detection...


### Student Exercise: The Automated News Aggregator
**Goal**: Build a pipeline that gathers news about "KPK Agriculture."
1. Use a **public API** (like NewsAPI) or a static site with **BeautifulSoup** to gather headlines.
2. Implement a **Rate Limiter** that waits 3 seconds between every 5 headlines.
3. Use a **Dictionary** to store the Title and the Link.
4. Save the final list as a `.json` file.
5. **Bonus**: Try to use **Selenium** to "Click" the 'Next' button on a page and scrape the second page of results.